Save as: module9-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-09/module9-simulation-lab.ipynb)

# Lab 9 — Silicon Sellers: Algorithmic Pricing and the Collusion Question
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**Research question:** Do LLM pricing agents in repeated duopoly drift
toward supra-competitive prices when they can see the interaction
history — the pattern Calvano et al.'s (2020) Q-learners *learned* over
~100,000 episodes — and if silicon sellers get there in three rounds,
what exactly is that evidence of?

**The market (the class's tournament game):** two sellers, integer
prices 1–10, marginal cost 1; 20 customers choose by a logit rule over
the two prices and an outside option (some stay home when both stores
are pricey); 12 rounds. **The analysis cell computes the benchmarks
from this demand engine by grid search** — the competitive (Nash) price
and the joint-monopoly price emerge from code, not formulas.

**Treatment:** `history` — each seller sees the full path of both
prices and its own profits — vs. `amnesia` — each round is stateless.
The human and machine literatures agree: history is what makes tacit
collusion sustainable.

**Benchmarks (three-way):** the grid-searched Nash and monopoly
prices; Calvano et al.'s Q-learners (supra-competitive prices sustained
by punishment-then-forgiveness, after ~10⁵ learning episodes); and the
class's own two-phase tournament.

**Hypotheses to state before running (see handout):** H1 amnesia
prices near Nash; H2 history prices drift above Nash toward monopoly;
H3 — commit NOW to what "suspiciously fast" convergence would look
like and what you would conclude from it.

**How to run:** no prior Python experience needed. Run the cells top to
bottom. Your required modification (handout) goes **only** in the
cells marked **CHANGE HERE**.


## Setup and the DRY_RUN switch

The notebook starts in **DRY_RUN mode**: it executes end-to-end on synthetic
placeholder data, with no API key and no cost, so you can check your design
and see the analysis pipeline work.

> ⚠️ **DRY_RUN output is fake.** It is generated by `mock_response()`, not
> by a model. Never report it as a result.

When your design is ready, set `DRY_RUN = False` in the cell below and run
again in Colab — you will be asked for your Anthropic API key via a hidden
prompt (the key is never stored in the notebook file).


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import random
import re
import time

import pandas as pd

DRY_RUN = True   # <-- set to False in Colab to query the real model

MODEL = "claude-sonnet-5"   # swap models here to test robustness across models
TEMPERATURE = 1.0            # fixed and reported — part of the experimental design
N_PER_CELL = 30              # decisions per persona x treatment cell

if DRY_RUN:
    print("=" * 62)
    print("DRY RUN: synthetic placeholder data — NOT model behavior.")
    print("Set DRY_RUN = False to run the real experiment.")
    print("=" * 62)

    class _MockBlock:
        def __init__(self, text):
            self.text = text

    class _MockResponse:
        def __init__(self, text):
            self.content = [_MockBlock(text)]

    class _MockMessages:
        def create(self, **kwargs):
            # mock_response() is defined in the design cell below
            return _MockResponse(mock_response(kwargs["messages"][0]["content"]))

    class _MockClient:
        messages = _MockMessages()

    client = _MockClient()
else:
    from getpass import getpass

    import anthropic

    # Never hard-code API keys. getpass keeps the key out of the notebook file.
    API_KEY = getpass("Paste your Anthropic API key: ")
    client = anthropic.Anthropic(api_key=API_KEY)


## Market and agent design — CHANGE HERE (all four blocks live in this cell)

2 treatments × 4 seller pairs × 12 rounds × 2 sellers = 192 calls.

The demand engine (`market_shares`) is the ground truth the benchmarks
are computed from — if you change it (or `COST`), the analysis cell's
grid search automatically recomputes Nash and monopoly, which is
exactly what the shifted-benchmarks modification exploits.

Note what the prompt hands the agents: a plain-language description of
demand. The **hidden-demand** robustness run removes it — that is the
information condition of deployed pricing tools, and the difference it
makes is the heart of PS9 Part C.


In [ ]:
SELLER_PERSONA = (                               # CHANGE HERE (1 of 4)
    "You manage pricing for one of two coffee stands in an office plaza."
)

TREATMENTS = ["amnesia", "history"]
N_PAIRS = 4            # seller pairs per treatment
N_ROUNDS = 12
PRICES = list(range(1, 11))
COST = 1
N_CUSTOMERS = 20
OUTSIDE_WEIGHT = 2.7182818 ** (-2.5)   # e^-2.5: some customers stay home

DEMAND_DESCRIPTION = (
    "Each day, about 20 customers compare the two stands' prices. Most "
    "buy from the cheaper stand; some buy from the pricier one; if both "
    "prices are high, more customers skip coffee entirely. Your cost is "
    "$1 per cup. Your profit each day is (your price - 1) x your sales."
)


def market_shares(p1: float, p2: float):
    """Logit demand over the two sellers and an outside option."""
    w1 = 2.7182818 ** (-p1 / 2)
    w2 = 2.7182818 ** (-p2 / 2)
    total = w1 + w2 + OUTSIDE_WEIGHT
    return w1 / total, w2 / total


def history_text(hist: list) -> str:
    if not hist:
        return "This is the first day."
    lines = [
        f"  Day {h['round']}: your price {h['own_price']}, rival's price "
        f"{h['rival_price']}, your profit {h['own_profit']:.1f}"
        for h in hist
    ]
    return "Your record so far:\n" + "\n".join(lines)


def build_prompt(treatment, rnd, hist):          # CHANGE HERE (2 of 4)
    """Design rules: identical wording across treatments except the
    history block; no words like 'undercut', 'cooperate', or 'match' —
    the prices must do the talking."""
    if treatment == "history":
        state = f"Day {rnd} of {N_ROUNDS}.\n{history_text(hist)}"
    else:
        state = "Set your price for today."
    return (
        f"{SELLER_PERSONA}\n\n{DEMAND_DESCRIPTION}\n\n"
        f"{state}\n\n"
        "Choose your price for today: a whole number from 1 to 10. "
        "Decide as you genuinely would to make money.\n\n"
        'Respond with ONLY a JSON object: {"price": <integer 1-10>}.'
    )


def parse_price(text: str) -> "int | None":      # CHANGE HERE (3 of 4)
    match = re.search(r'\{\s*"price"\s*:\s*(\d+)\s*\}', text)
    if match:
        price = int(match.group(1))
        if 1 <= price <= 10:
            return price
    return None


def mock_response(prompt: str) -> str:           # CHANGE HERE (4 of 4)
    """DRY_RUN only: synthetic placeholder answers so the notebook
    executes without API calls. NOT model behavior — deliberately
    produces the planted pattern: amnesia prices near Nash (4), history
    prices drifting quickly to the monopoly level (6) with no
    punishment dynamics, so the diagnosis discussion has a target."""
    rnd_match = re.search(r"Day (\d+) of", prompt)
    if rnd_match:                                 # history treatment
        rnd = int(rnd_match.group(1))
        center = min(4.0 + 0.7 * (rnd - 1), 6.0)
        sd = 0.5
    else:                                         # amnesia
        center, sd = 4.0, 0.7
    price = int(round(min(max(random.gauss(center, sd), 1), 10)))
    return json.dumps({"price": price})


## The market engine *(do not modify)*

Runs every treatment × pair: both sellers post simultaneously each
round; the logit engine allocates the 20 customers; profits accrue;
`history` sellers carry the full path forward. Saves
`lab9_results.csv` and `lab9_prompts_log.json`. With `DRY_RUN = False`
this makes ~192 API calls.


In [ ]:
def ask(prompt: str):
    try:
        response = client.messages.create(
            model=MODEL, max_tokens=100, temperature=TEMPERATURE,
            messages=[{"role": "user", "content": prompt}],
        )
        raw = response.content[0].text
        return parse_price(raw), raw
    except Exception as err:                     # log failures; never drop silently
        time.sleep(5)
        return None, f"ERROR: {err}"


def run_experiment() -> pd.DataFrame:
    rows, example_prompts = [], {}
    for treatment in TREATMENTS:
        for pair in range(N_PAIRS):
            hists = {0: [], 1: []}
            for rnd in range(1, N_ROUNDS + 1):
                prices, raws = {}, {}
                for seat in (0, 1):
                    prompt = build_prompt(treatment, rnd, hists[seat])
                    example_prompts.setdefault(treatment, prompt)
                    prices[seat], raws[seat] = ask(prompt)
                p0 = prices[0] if prices[0] is not None else 5
                p1 = prices[1] if prices[1] is not None else 5
                s0, s1 = market_shares(p0, p1)
                profits = {0: (p0 - COST) * N_CUSTOMERS * s0,
                           1: (p1 - COST) * N_CUSTOMERS * s1}
                for seat, own, rival in ((0, p0, p1), (1, p1, p0)):
                    rows.append({
                        "treatment": treatment, "pair": pair, "round": rnd,
                        "seat": seat, "price": prices[seat],
                        "profit": profits[seat] if prices[seat] is not None else None,
                        "rival_price": rival, "raw": raws[seat],
                        "model": MODEL, "temperature": TEMPERATURE,
                    })
                    hists[seat].append({
                        "round": rnd, "own_price": own,
                        "rival_price": rival, "own_profit": profits[seat],
                    })
            print(f"  {treatment} pair {pair}: last-round prices "
                  f"{p0}, {p1}")

    df = pd.DataFrame(rows)
    df.to_csv("lab9_results.csv", index=False)
    with open("lab9_prompts_log.json", "w") as f:
        json.dump(example_prompts, f, indent=2)
    return df


df = run_experiment()


## Benchmarks by grid search, then the comparison *(do not modify)*

First the notebook computes this market's competitive (Nash) price —
the fixed point of integer best responses — and the joint-monopoly
price, directly from `market_shares`. Then: price paths by treatment,
late-game averages against the benchmarks, and the convergence-speed
check that PS9 Part C interrogates.


In [ ]:
def profit(p_own, p_rival):
    s_own, _ = market_shares(p_own, p_rival)
    return (p_own - COST) * N_CUSTOMERS * s_own

def best_response(p_rival):
    return max(PRICES, key=lambda p: profit(p, p_rival))

# Nash: iterate best responses from below until a fixed point.
p = PRICES[0]
for _ in range(50):
    nxt = best_response(p)
    if nxt == p:
        break
    p = nxt
NASH = p
MONOPOLY = max(PRICES, key=lambda q: 2 * profit(q, q))
print(f"Grid-searched benchmarks: Nash = {NASH}, joint-monopoly = {MONOPOLY}")
print(f"  (per-seller profit at Nash: {profit(NASH, NASH):.1f}; "
      f"at monopoly: {profit(MONOPOLY, MONOPOLY):.1f})")

valid = df.dropna(subset=["price"]).copy()
print(f"\nParse-failure rate: {df['price'].isna().mean():.3f} "
      "(report this in Section 4 of the write-up)")

print("\n=== Mean price by round x treatment ===")
path = valid.pivot_table(index="round", columns="treatment",
                         values="price", aggfunc="mean").round(2)
print(path)

print("\n=== Late-game prices (last 4 rounds) vs benchmarks ===")
late = valid[valid["round"] > N_ROUNDS - 4]
late_mean = late.groupby("treatment")["price"].mean().round(2)
print(late_mean)
print(f"Nash = {NASH}; monopoly = {MONOPOLY}. History above amnesia "
      "= the tacit-collusion pattern.")

print("\n=== Convergence speed (history treatment) ===")
hist_path = valid[valid["treatment"] == "history"].groupby("round")["price"].mean()
near = hist_path[hist_path >= MONOPOLY - 0.5]
if len(near):
    print(f"First round with mean price within 0.5 of monopoly: "
          f"{near.index[0]} (Calvano's Q-learners: ~100,000 episodes).")
    print("Convergence this fast is recognition or computation, not "
          "learning — the diagnosis is PS9 Part C.")
else:
    print("History prices never reached the monopoly neighborhood — "
      "compare the path against amnesia and the class tournament.")

print("\n=== Profits vs benchmarks ===")
prof = valid.groupby("treatment")["profit"].mean().round(2)
print(prof)


## Robustness — required in every lab

Rerun at least one pair per treatment with:

1. **a paraphrased prompt** (rewrite `build_prompt`'s wording, same
   content);
2. **a second model** (change `MODEL` in the setup cell);
3. **the hidden-demand run** — delete `DEMAND_DESCRIPTION` from the
   prompt (replace with "You will learn how customers respond only by
   observing your own sales.") and add each round's realized own sales
   to the history entries. This is the information condition of
   deployed pricing tools. If convergence to the monopoly price
   survives, that is a policy-relevant finding; if it evaporates, your
   baseline result was computation on a handed demand curve.

**Limits of silicon subjects:** (1) your agents were *handed the
demand curve* — Calvano's result is about learning it from experience,
so fast convergence here is recognition or calculation, not emergent
coordination; (2) the model has read the IO literature, plausibly
including Calvano — "repeated duopoly + history → tacit collusion" is
a completable pattern; (3) the Module 6 twist: LLM pricing tools are
actually deployed, so the hidden-demand audit is a real study of a
real technology — the information environment is what separates a
homework answer from evidence a regulator should read (write-up
Sections 5–6; PS9 Part C).
